### Query Decomposition
Split a complex question into simple sub-questions → retrieve and answer each → combine into one final answer.

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

C:\Users\Adit Sharma\AppData\Local\Temp\ipykernel_30640\1565552403.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("groq_api")

llm = init_chat_model("groq:openai/gpt-oss-20b")
llm.invoke("Say hi in 3 words").content      

'Hello there friend.'

In [4]:
loader = TextLoader("langchain_crewai_dataset.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(len(chunks))     

241


In [5]:
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
query = "How does LangChain use memory and agents compared to CrewAI?"
for d in retriever.invoke(query):
    print("-", d.page_content[:100])

- CrewAI is compatible with LangChain agents and tools, allowing hybrid systems where LangChain handle
- LangChain agents operate using a planner-executor model, where the agent plans out a sequence of too
- CrewAI is compatible with LangChain agents and tools, allowing hybrid systems where LangChain handle
- CrewAI is compatible with LangChain agents and tools, allowing hybrid systems where LangChain handle


In [7]:
decomposition_prompt = PromptTemplate.from_template("""
Break the question below into 2 to 4 simple sub-questions. Each sub-question must cover one topic only.
Return ONLY the sub-questions, one per line. No numbering, no bullets, no explanations, no intro text.

Question: {question}
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [8]:
def decompose(question):
    text = decomposition_chain.invoke({"question": question})
    return [line.strip("-•*0123456789. ").strip()
            for line in text.splitlines()
            if line.strip().endswith("?")]

sub_questions = decompose(query)
sub_questions


['How does LangChain implement memory in its agent framework?',
 'How does CrewAI handle memory for its agents?',
 "What are the key differences between LangChain's and CrewAI's approaches to memory and agents?"]

In [9]:
qa_prompt = PromptTemplate.from_template("""
Answer using ONLY the context. If the context doesn't contain the answer, say "Not in the documents."

Context:
{context}

Question: {input}
""")
qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

# test on the first sub-question
subq = sub_questions[0]
qa_chain.invoke({"input": subq, "context": retriever.invoke(subq)})


'Not in the documents.'

In [10]:
synthesis_prompt = PromptTemplate.from_template("""
Using the sub-answers below, write one clear answer to the original question.
Don't add facts that aren't in the sub-answers.

Original question: {question}

Sub-answers:
{sub_answers}
""")
synthesis_chain = synthesis_prompt | llm | StrOutputParser()


In [11]:
def decomposition_rag(question):
    sub_questions = decompose(question)
    pairs = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        answer = qa_chain.invoke({"input": subq, "context": docs})
        pairs.append(f"Q: {subq}\nA: {answer}")
    sub_answers = "\n\n".join(pairs)
    final = synthesis_chain.invoke({"question": question, "sub_answers": sub_answers})
    return sub_questions, sub_answers, final


In [12]:
subs, sub_answers, final = decomposition_rag(query)
print("🧩 Sub-questions:", *subs, sep="\n- ")
print("\n📄 Sub-answers:\n", sub_answers)
print("\n✅ Final answer:\n", final)


🧩 Sub-questions:
- How does LangChain implement memory management within its framework?
- What kinds of agents does LangChain support and how are they used?
- How does CrewAI differ from LangChain in memory handling and agent usage?

📄 Sub-answers:
 Q: How does LangChain implement memory management within its framework?
A: LangChain implements memory management through dedicated memory modules such as **ConversationBufferMemory** and **ConversationSummaryMemory**. These modules allow the LLM to keep track of previous conversation turns or to summarize long interactions so that the token limits can be respected.

Q: What kinds of agents does LangChain support and how are they used?
A: LangChain supports **agent-based workflows** that let an LLM reason about which external tool to call, what input to give that tool, and how to interpret the tool’s response. These agents can:

* **Plan** a sequence of tool invocations (the “planner” part) to reach a goal.
* **Execute** those calls (the “e